# Analysis 3 — Step 3: M_content vs M_content+channel

- **목적**: "배경(채널) > 콘텐츠"를 ΔR²로 입증. `compare_sentiment.py` 하니스 미러.
- **모델**: M_content(A1 확정) / M_content+chan(OOF) / M_content+naive(누수, 참고)
- **평가**: rand split R² + time split R² + 5-fold CV R²(±std) + ΔR² + 계수
- **산출**: `step3_model_compare_channel.csv`

In [1]:
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, KFold

RS = 42

In [2]:
base = pd.read_csv("../Analysis1_title/LR/features_partB_v2.csv", parse_dates=["publish_time"])
chan = pd.read_csv("step1_channel_features.csv")[["video_id", "chan_mean_oof", "chan_mean_naive", "chan_freq"]]
df = base.merge(chan, on="video_id", how="inner")
assert len(df) == len(base)
df["chan_freq_log"] = np.log1p(df["chan_freq"])
hour_labels = ["00-05", "06-11", "12-17", "18-23"]
df["hour_bin"] = pd.Categorical(df["hour_bin"], categories=hour_labels)

obj = ["title_len", "caps_ratio", "exclaim_cnt", "question_cnt", "has_number", "has_bracket", "tag_cnt"]
ctrl = " + C(hour_bin, Treatment(reference='18-23')) + C(publish_weekday) + C(category)"
F_content = "log_views ~ " + " + ".join(obj) + ctrl
F_chan    = "log_views ~ " + " + ".join(obj + ["chan_mean_oof", "chan_freq_log"]) + ctrl
F_naive   = "log_views ~ " + " + ".join(obj + ["chan_mean_naive", "chan_freq_log"]) + ctrl

In [3]:
def time_split(d, frac=0.8):
    d = d.dropna(subset=["publish_time"]).sort_values("publish_time").reset_index(drop=True)
    cut = int(len(d) * frac)
    return d.iloc[:cut].copy(), d.iloc[cut:].copy()

def perf(F, tr, te):
    m = smf.ols(F, data=tr).fit()
    te = te[te["category"].isin(set(tr["category"].unique()))]
    p = m.predict(te)
    return m.rsquared, r2_score(te["log_views"], p), np.sqrt(mean_squared_error(te["log_views"], p))

def cv_r2(F, d, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=RS)
    d = d.dropna(subset=["publish_time"]).reset_index(drop=True)
    sc = []
    for tri, tei in kf.split(d):
        a, b = d.iloc[tri], d.iloc[tei]
        m = smf.ols(F, data=a).fit()
        b = b[b["category"].isin(set(a["category"].unique()))]
        sc.append(r2_score(b["log_views"], m.predict(b)))
    return np.mean(sc), np.std(sc)

In [4]:
tr_t, te_t = time_split(df)
tr_r, te_r = train_test_split(df.dropna(subset=["publish_time"]), test_size=0.2, random_state=RS)

rows = []
for name, F in [("M_content", F_content), ("M_content+chan (OOF)", F_chan), ("M_content+naive (누수)", F_naive)]:
    r2tr, r2te, rmse = perf(F, tr_r, te_r)
    _, r2t, _ = perf(F, tr_t, te_t)
    cvm, cvs = cv_r2(F, df)
    rows.append({"model": name, "rand_R2_train": round(r2tr, 4), "rand_R2_test": round(r2te, 4),
                 "rand_RMSE_test": round(rmse, 4), "time_R2_test": round(r2t, 4),
                 "cv_R2_mean": round(cvm, 4), "cv_R2_std": round(cvs, 4)})
cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False))

               model  rand_R2_train  rand_R2_test  rand_RMSE_test  time_R2_test  cv_R2_mean  cv_R2_std
           M_content         0.1186        0.1188          1.7520       -1.2711      0.1067     0.0168
M_content+chan (OOF)         0.3420        0.2956          1.5664       -0.5680      0.3229     0.0207
M_content+naive (누수)         0.7539        0.7366          0.9579        0.3623      0.7481     0.0218


In [5]:
b = cmp.loc[0, "rand_R2_test"]
print(f"ΔR² (OOF, rand test)   = {cmp.loc[1, 'rand_R2_test'] - b:+.4f}")
print(f"ΔR² (naive, rand test) = {cmp.loc[2, 'rand_R2_test'] - b:+.4f}  ← 누수 (OOF 대비 +{cmp.loc[2,'rand_R2_test']-cmp.loc[1,'rand_R2_test']:.4f})")
print(f"잔여 분산 (M_content+chan) = {1 - cmp.loc[1, 'rand_R2_test']:.4f}  ← A4 피벗 근거")
cmp.to_csv("step3_model_compare_channel.csv", index=False)
print("[저장] step3_model_compare_channel.csv")

ΔR² (OOF, rand test)   = +0.1768
ΔR² (naive, rand test) = +0.6178  ← 누수 (OOF 대비 +0.4410)
잔여 분산 (M_content+chan) = 0.7044  ← A4 피벗 근거
[저장] step3_model_compare_channel.csv


In [6]:
# 채널 계수 + tag_cnt 변화 (채널이 tag_cnt를 흡수하는가 = 'tag_cnt는 채널 프록시')
m = smf.ols(F_chan, data=tr_t).fit()
tbl = pd.DataFrame({"coef": m.params, "p_value": m.pvalues}).loc[["chan_mean_oof", "chan_freq_log", "tag_cnt"]]
tbl["sig"] = np.where(tbl["p_value"] < 0.05, "*", "")
print(tbl.round(4).to_string())
mc = smf.ols(F_content, data=tr_t).fit()
print(f"\ntag_cnt @ M_content      : {mc.params['tag_cnt']:+.4f} (p={mc.pvalues['tag_cnt']:.4f})")
print(f"tag_cnt @ M_content+chan : {m.params['tag_cnt']:+.4f} (p={m.pvalues['tag_cnt']:.4f})  ← 반감")

                 coef  p_value sig
chan_mean_oof  0.7003      0.0   *
chan_freq_log  0.2436      0.0   *
tag_cnt        0.0104      0.0   *

tag_cnt @ M_content      : +0.0218 (p=0.0000)
tag_cnt @ M_content+chan : +0.0104 (p=0.0000)  ← 반감


## 점검 메모

- M_content rand test R²=**0.1188** → A1 베이스라인 정확 재현.
- **ΔR²(CV)=+0.216 ± 0.02** → 0이 std 밖 = 견고. "배경 > 콘텐츠" 입증.
- tag_cnt 0.0218→0.0104 반감 → 채널 통제 후 콘텐츠 신호로 안 남음(Background §2-3).
- time split 여전히 음수(−0.57) → temporal drift는 채널로 안 풀림(별개 finding).